# Azure AI Foundry Agent - 교수 페르소나 QA 에이전트 Quickstart

`pdf_qa`의 `AzureFoundryProvider(mode="agent")`가 내부적으로 사용하는
**Foundry Agent Service** 흐름을 그대로 보여줍니다.


## 사전 준비
1. Azure AI Foundry 프로젝트 엔드포인트 (`AZURE_AI_PROJECT_ENDPOINT`)
2. 모델 배포 (예: `gpt-4o`)
3. `az login` + 역할: **Azure AI User** (프로젝트 대상)


In [ ]:
%pip install -U azure-ai-projects azure-ai-agents azure-identity

In [ ]:
import os
os.environ.setdefault("AZURE_AI_PROJECT_ENDPOINT", "https://<proj>.services.ai.azure.com/api/projects/<name>")
os.environ.setdefault("AZURE_AI_AGENT_MODEL", "gpt-4o")

## 1. 에이전트 생성 → 스레드 실행 → 응답 파싱 (원시 SDK)

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from azure.ai.agents.models import ListSortOrder

project = AIProjectClient(
    endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential(),
)
agents = project.agents

agent = agents.create_agent(
    model=os.environ["AZURE_AI_AGENT_MODEL"],
    name="pdf-qa-professor",
    instructions=(
        "You are a meticulous university professor who writes exam questions. "
        "Follow the user's formatting instructions exactly and always answer in Korean, "
        "returning only the requested JSON block."
    ),
)
print("agent:", agent.id)

thread = agents.threads.create()
agents.messages.create(
    thread_id=thread.id,
    role="user",
    content=(
        "Context: 금융보안교육센터의 주소는 서울특별시 영등포구 의사당대로 143 입니다.\n"
        "위 문맥으로 한국어 QA 2개를 ```json {\"QUESTION\":..., \"ANSWER\":...}``` 형식으로 생성하세요."
    ),
)
run = agents.runs.create_and_process(thread_id=thread.id, agent_id=agent.id)
print("run status:", run.status)

for m in agents.messages.list(thread_id=thread.id, order=ListSortOrder.ASCENDING):
    if m.role == "assistant" and m.text_messages:
        print(m.text_messages[-1].text.value)

## 2. 동일 로직을 `pdf_qa` 공급자로 호출

`AZURE_MODE=agent`로 두면 파이프라인 전체가 Foundry Agent로 동작합니다.

In [ ]:
import sys
sys.path.insert(0, "../pdf_qa_extraction")
os.environ["LLM_PROVIDER"] = "azure"
os.environ["AZURE_MODE"] = "agent"

from pdf_qa import QAConfig, get_provider

provider = get_provider("azure", config=QAConfig(domain="International Finance"))
qa = provider.generate_text_qa("금융보안원은 서울에 위치한 기관입니다.", "International Finance", "2")
print(qa)

## 3. 정리 (에이전트 삭제)

In [ ]:
agents.delete_agent(agent.id)
print("deleted:", agent.id)